# Pair 3: Profile Attack Types (Clustering)

**Objective:** Group the 11,743 anomaly rows from the Kaggle "Network Intrusion Detection" dataset into attack families (floods, scans, brute-force, etc.) so analysts know what kind of attack they're facing. Map each cluster to a MITRE ATT&CK technique as a hypothesis.

---

## 1. Imports & Setup

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_index
)
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print(f'pandas {pd.__version__} | scikit-learn {sklearn.__version__} | matplotlib {matplotlib.__version__}')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.axisbelow': True,
    'figure.figsize': (10, 6)
})

COLORS = {'normal': '#2a78d6', 'anomaly': '#eb6834'}

: 

## 2. Load & Verify Data

In [ ]:
train = pd.read_csv('Train_data.csv')
test = pd.read_csv('Test_data.csv')

print(f'Train_data.csv: {train.shape[0]:,} rows, {train.shape[1]} columns')
print(f'Test_data.csv:  {test.shape[0]:,} rows, {test.shape[1]} columns')

# Verify class balance
balance = train['class'].value_counts()
print(f'\nClass balance:\n{balance}')
print(f"Anomaly share: {balance.get('anomaly', 0) / len(train):.1%}")

## 3. Filter to Anomalies Only (Clustering Target)

In [ ]:
# Use ONLY anomaly rows for clustering
anomalies = train[train['class'] == 'anomaly'].copy()
print(f'Anomaly rows for clustering: {len(anomalies):,}')

# Drop label and constant columns
constant_cols = ['num_outbound_cmds', 'is_host_login']
features = [c for c in anomalies.columns if c not in ['class'] + constant_cols]
X = anomalies[features].copy()

print(f'Features for clustering: {len(features)}')
print(f'Constant columns dropped: {constant_cols}')

# Categorical vs numeric
categorical = ['protocol_type', 'service', 'flag']
numeric = [c for c in features if c not in categorical]
print(f'Categorical: {categorical}')
print(f'Numeric: {len(numeric)} features')

## 4. Preprocessing Pipeline

- **Skewed numeric** (`duration`, `src_bytes`, `dst_bytes`) → `log1p` → `StandardScaler`
- **Other numeric** → `StandardScaler`
- **Categorical** → `OneHotEncoder(handle_unknown='ignore')`
- **Rare services** grouped into `"other"` to limit cardinality

In [ ]:
# Group rare services into 'other' (threshold: < 100 occurrences)
service_counts = X['service'].value_counts()
rare_services = service_counts[service_counts < 100].index.tolist()
print(f'Rare services to group (count < 100): {len(rare_services)}')
print(f'Examples: {rare_services[:10]}')

X['service_grouped'] = X['service'].replace(rare_services, 'other')

# Update feature lists
categorical_grouped = ['protocol_type', 'service_grouped', 'flag']
numeric = [c for c in features if c not in categorical]
skewed = ['duration', 'src_bytes', 'dst_bytes']
numeric_regular = [c for c in numeric if c not in skewed]

print(f'Skewed numeric: {skewed}')
print(f'Regular numeric: {len(numeric_regular)} features')
print(f'Categorical (grouped): {categorical_grouped}')

In [ ]:
from sklearn.pipeline import Pipeline

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_grouped),
    ('num_skewed', Pipeline([
        ('log', FunctionTransformer(np.log1p, validate=False)),
        ('scale', StandardScaler())
    ]), skewed),
    ('num_regular', StandardScaler(), numeric_regular)
], remainder='drop')

X_processed = preprocess.fit_transform(X)
print(f'Processed shape: {X_processed.shape}')

# Feature names for interpretation
cat_names = preprocess.named_transformers_['cat'].get_feature_names_out(categorical_grouped)
feature_names = list(cat_names) + skewed + numeric_regular
print(f'Total features after encoding: {len(feature_names)}')

## 5. PCA for Visualization & Dimensionality Reduction

In [ ]:
# PCA for 2D visualization
pca_2d = PCA(n_components=2, random_state=SEED)
X_2d = pca_2d.fit_transform(X_processed)
print(f'2D PCA explained variance: {pca_2d.explained_variance_ratio_.sum():.1%}')

# PCA for clustering: keep 95% variance
pca_95 = PCA(n_components=0.95, random_state=SEED)
X_pca = pca_95.fit_transform(X_processed)
print(f'PCA components for 95% variance: {pca_95.n_components_}')
print(f'Explained variance ratio (cumulative): {pca_95.explained_variance_ratio_.sum():.1%}')

In [ ]:
# 2D scatter plot (colored by... nothing yet, just structure)
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], s=3, alpha=0.5, c='#eb6834')
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
ax.set_title('Anomaly Data: PCA 2D Projection (before clustering)')
fig.tight_layout()
plt.show()

## 6. K-Means: Elbow Method & Silhouette Analysis

In [ ]:
K_range = range(2, 16)
inertias = []
silhouettes = []

# Sample for silhouette (O(n^2) complexity)
sample_size = min(5000, len(X_pca))
sample_idx = np.random.choice(len(X_pca), sample_size, replace=False)
X_sample = X_pca[sample_idx]

for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    
    # Silhouette on sample
    sample_labels = labels[sample_idx]
    sil = silhouette_score(X_sample, sample_labels)
    silhouettes.append(sil)
    print(f'k={k:2d}  inertia={km.inertia_:.2e}  silhouette={sil:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(K_range, inertias, 'o-', color='#eb6834')
ax1.set_xlabel('k')
ax1.set_ylabel('Inertia')
ax1.set_title('K-Means Elbow Method')
ax1.grid(True, alpha=0.3)

ax2.plot(K_range, silhouettes, 'o-', color='#2a78d6')
ax2.set_xlabel('k')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('K-Means Silhouette Score (sampled)')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 7. Gaussian Mixture Model: BIC/AIC Selection

In [ ]:
bics = []
aics = []

for k in K_range:
    gmm = GaussianMixture(n_components=k, random_state=SEED, n_init=5)
    gmm.fit(X_pca)
    bics.append(gmm.bic(X_pca))
    aics.append(gmm.aic(X_pca))
    print(f'k={k:2d}  BIC={gmm.bic(X_pca):.2e}  AIC={gmm.aic(X_pca):.2e}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(K_range, bics, 'o-', label='BIC', color='#eb6834')
ax.plot(K_range, aics, 's-', label='AIC', color='#2a78d6')
ax.set_xlabel('Number of components (k)')
ax.set_ylabel('Score')
ax.set_title('GMM: BIC & AIC for Component Selection')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 8. DBSCAN: eps Estimation via k-Distance Plot

In [ ]:
# k-distance plot (k = n_components + 1)
k_neighbors = pca_95.n_components_ + 1
neighbors = NearestNeighbors(n_neighbors=k_neighbors)
neighbors_fit = neighbors.fit(X_pca)
distances, _ = neighbors_fit.kneighbors(X_pca)
distances = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(distances, color='#eb6834', linewidth=0.5)
ax.set_xlabel('Points sorted by distance')
ax.set_ylabel(f'{k_neighbors}-NN Distance')
ax.set_title(f'k-Distance Plot for DBSCAN eps Selection (k={k_neighbors})')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f'k = {k_neighbors}')
print(f'Distance range: {distances.min():.3f} - {distances.max():.3f}')
print(f'Median: {np.median(distances):.3f}, 90th percentile: {np.percentile(distances, 90):.3f}')

In [ ]:
# Try a few eps values around the knee
eps_candidates = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0]
min_samples = k_neighbors

for eps in eps_candidates:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_pca)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_outliers = np.sum(labels == -1)
    outlier_pct = n_outliers / len(labels) * 100
    print(f'eps={eps:.1f}  clusters={n_clusters}  outliers={n_outliers:,} ({outlier_pct:.1f}%)')

## 9. Model Selection & Final Clustering

Based on the analyses above, choose the best k for K-Means and GMM, and best eps for DBSCAN. Then run final models and compute metrics.

In [ ]:
# ============================================================
# TODO: Set these based on elbow/silhouette/BIC/k-distance plots above
# ============================================================
CHOSEN_K = 6      # <-- UPDATE after reviewing plots
CHOSEN_EPS = 1.0  # <-- UPDATE after reviewing k-distance plot

print(f'Selected K-Means k: {CHOSEN_K}')
print(f'Selected GMM k: {CHOSEN_K}')
print(f'Selected DBSCAN eps: {CHOSEN_EPS}, min_samples: {min_samples}')

In [ ]:
# --- K-Means ---
kmeans = KMeans(n_clusters=CHOSEN_K, random_state=SEED, n_init=10)
km_labels = kmeans.fit_predict(X_pca)

# --- GMM ---
gmm = GaussianMixture(n_components=CHOSEN_K, random_state=SEED, n_init=5)
gmm.fit(X_pca)
gmm_labels = gmm.predict(X_pca)

# --- DBSCAN ---
dbscan = DBSCAN(eps=CHOSEN_EPS, min_samples=min_samples)
db_labels = dbscan.fit_predict(X_pca)
db_n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
db_n_outliers = np.sum(db_labels == -1)

print('=== Cluster Counts ===')
print(f'K-Means: {CHOSEN_K} clusters')
print(f'GMM:     {CHOSEN_K} clusters')
print(f'DBSCAN:  {db_n_clusters} clusters, {db_n_outliers:,} outliers ({db_n_outliers/len(db_labels)*100:.1f}%)')

print('\n=== Cluster Sizes ===')
for name, labels in [('K-Means', km_labels), ('GMM', gmm_labels), ('DBSCAN', db_labels)]:
    unique, counts = np.unique(labels[labels >= 0], return_counts=True)
    print(f'{name}: {dict(zip(unique, counts))}')

## 10. Clustering Metrics

In [ ]:
def compute_metrics(X, labels, name):
    """Compute clustering metrics (silhouette on sample for speed)."""
    valid = labels >= 0
    if valid.sum() < 2:
        return {}
    X_valid = X[valid]
    labels_valid = labels[valid]
    
    # Sample for silhouette
    sample_size = min(5000, len(X_valid))
    idx = np.random.choice(len(X_valid), sample_size, replace=False)
    
    sil = silhouette_score(X_valid[idx], labels_valid[idx])
    db = davies_bouldin_score(X_valid, labels_valid)
    ch = calinski_harabasz_score(X_valid, labels_valid)
    
    return {
        'method': name,
        'n_clusters': len(np.unique(labels_valid)),
        'silhouette': sil,
        'davies_bouldin': db,
        'calinski_harabasz': ch
    }

metrics = []
metrics.append(compute_metrics(X_pca, km_labels, 'K-Means'))
metrics.append(compute_metrics(X_pca, gmm_labels, 'GMM'))
metrics.append(compute_metrics(X_pca, db_labels, 'DBSCAN'))

metrics_df = pd.DataFrame(metrics)
display(metrics_df.round(4))

## 11. Stability Analysis (Adjusted Rand Index across seeds)

In [ ]:
def stability_analysis(model_class, model_params, X, n_runs=10, base_labels=None):
    """Rerun clustering with different seeds, compute ARI vs base."""
    if base_labels is None:
        base_model = model_class(**model_params, random_state=SEED)
        base_labels = base_model.fit_predict(X)
    
    ari_scores = []
    for seed in range(n_runs):
        model = model_class(**model_params, random_state=seed)
        labels = model.fit_predict(X)
        ari = adjusted_rand_index(base_labels, labels)
        ari_scores.append(ari)
    
    return np.array(ari_scores), base_labels

print('Running stability analysis (10 seeds each)...')

km_ari, km_base = stability_analysis(KMeans, {'n_clusters': CHOSEN_K, 'n_init': 10}, X_pca)
gmm_ari, gmm_base = stability_analysis(GaussianMixture, {'n_components': CHOSEN_K, 'n_init': 5}, X_pca)

print(f'K-Means ARI:  mean={km_ari.mean():.4f}  std={km_ari.std():.4f}  min={km_ari.min():.4f}  max={km_ari.max():.4f}')
print(f'GMM ARI:      mean={gmm_ari.mean():.4f}  std={gmm_ari.std():.4f}  min={gmm_ari.min():.4f}  max={gmm_ari.max():.4f}')

# DBSCAN stability (no random_state, but we can jitter data slightly or just note it's deterministic)
print('\nDBSCAN is deterministic (no random_state); stability not applicable.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([km_ari, gmm_ari], labels=['K-Means', 'GMM'])
ax.set_ylabel('Adjusted Rand Index (vs seed=42)')
ax.set_title('Clustering Stability Across Random Seeds')
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
plt.show()

## 12. Cluster Profiling & Interpretation

Use the **best model** (based on metrics + stability) to profile each cluster by its average features.

In [ ]:
# Choose best model (example: K-Means if it has good metrics & stability)
BEST_MODEL = 'K-Means'  # <-- UPDATE after reviewing metrics
best_labels = km_labels if BEST_MODEL == 'K-Means' else gmm_labels

# Add cluster labels to anomaly dataframe
anomalies_labeled = anomalies.copy()
anomalies_labeled['cluster'] = best_labels

# Key features for profiling
profile_features = [
    'duration', 'src_bytes', 'dst_bytes',
    'protocol_type', 'service', 'flag',
    'count', 'srv_count', 'serror_rate', 'rerror_rate',
    'same_srv_rate', 'diff_srv_rate',
    'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'num_failed_logins', 'logged_in', 'is_guest_login',
    'num_compromised', 'root_shell', 'su_attempted',
    'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'hot', 'num_failed_logins'
]

# Compute cluster profiles (median for numeric, mode for categorical)
numeric_profile = [f for f in profile_features if f in anomalies_labeled.select_dtypes(include='number').columns]
categorical_profile = [f for f in profile_features if f in anomalies_labeled.select_dtypes(exclude='number').columns]

cluster_profiles = anomalies_labeled.groupby('cluster').agg({
    **{f: 'median' for f in numeric_profile},
    **{f: lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'N/A' for f in categorical_profile}
})

print(f'Cluster profiles shape: {cluster_profiles.shape}')
display(cluster_profiles.T)

In [ ]:
# Normalized heatmap of numeric features
numeric_cols = cluster_profiles.select_dtypes(include='number').columns
profile_numeric = cluster_profiles[numeric_cols]

# Z-score normalize per feature for heatmap
profile_normalized = (profile_numeric - profile_numeric.mean()) / profile_numeric.std()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(profile_normalized.T, cmap='RdBu_r', center=0, 
            annot=False, fmt='.1f', cbar_kws={'label': 'Z-score'},
            ax=ax)
ax.set_title(f'Cluster Profiles (Z-scored) — {BEST_MODEL}')
ax.set_xlabel('Cluster')
ax.set_ylabel('Feature')
fig.tight_layout()
plt.show()

In [ ]:
# PCA visualization colored by cluster
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scatter1 = axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=best_labels, s=3, alpha=0.6, cmap='tab20')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
axes[0].set_title(f'Anomaly Clusters (PCA 2D) — {BEST_MODEL}')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Also show on PCA-reduced space (first 2 of 95% components)
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=best_labels, s=3, alpha=0.6, cmap='tab20')
axes[1].set_xlabel('PC1 (95% variance space)')
axes[1].set_ylabel('PC2 (95% variance space)')
axes[1].set_title(f'Anomaly Clusters (PCA 95% space) — {BEST_MODEL}')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

fig.tight_layout()
plt.show()

## 13. MITRE ATT&CK Mapping (Hypothesis)

Based on cluster profiles, hypothesize the attack type and map to MITRE ATT&CK. **This is a hypothesis only — the dataset has no attack-type labels.**

In [ ]:
# ============================================================
# TODO: Fill in based on cluster profile interpretation
# ============================================================

mitre_mapping = {
    0: {
        'attack_type': 'SYN Flood / DoS',
        'mitre_technique': 'T1499 (Endpoint DoS) / T1498 (Network DoS)',
        'key_indicators': 'High serror_rate, flag=S0, high count/srv_count',
        'confidence': 'Medium'
    },
    1: {
        'attack_type': 'Brute Force / Credential Stuffing',
        'mitre_technique': 'T1110 (Brute Force) / T1110.004 (Credential Stuffing)',
        'key_indicators': 'High num_failed_logins, low logged_in, service=ssh/http',
        'confidence': 'High'
    },
    2: {
        'attack_type': 'Port/Service Scanning',
        'mitre_technique': 'T1595 (Active Scanning) / T1046 (Network Service Scanning)',
        'key_indicators': 'Many distinct services from one source, low bytes',
        'confidence': 'Medium'
    },
    3: {
        'attack_type': 'Exploit / Privilege Escalation',
        'mitre_technique': 'T1190 (Exploit Public-Facing App) / T1068 (Exploitation for Privilege Escalation)',
        'key_indicators': 'High root_shell, su_attempted, num_shells',
        'confidence': 'Low-Medium'
    },
    4: {
        'attack_type': 'Payload Deployment / Tool Transfer',
        'mitre_technique': 'T1105 (Ingress Tool Transfer) / T1059 (Command & Scripting)',
        'key_indicators': 'High num_file_creations, num_access_files',
        'confidence': 'Low'
    },
    5: {
        'attack_type': 'Unknown / Mixed',
        'mitre_technique': 'Multiple / Unclear',
        'key_indicators': 'Mixed features, no clear pattern',
        'confidence': 'Low'
    }
}

print('=== MITRE ATT&CK Mapping (HYPOTHESIS ONLY) ===')
for cluster_id, info in mitre_mapping.items():
    size = (best_labels == cluster_id).sum()
    print(f'\nCluster {cluster_id} ({size:,} rows, {size/len(best_labels)*100:.1f}%)')
    for k, v in info.items():
        print(f'  {k}: {v}')

## 14. Save Results

In [ ]:
# Save cluster assignments
output_df = anomalies[['protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes']].copy()
output_df['cluster'] = best_labels
output_df.to_csv('pair3_cluster_assignments.csv', index=False)
print('Saved: pair3_cluster_assignments.csv')

# Save cluster profiles
cluster_profiles.to_csv('pair3_cluster_profiles.csv')
print('Saved: pair3_cluster_profiles.csv')

# Save MITRE mapping
import json
with open('pair3_mitre_mapping.json', 'w') as f:
    json.dump(mitre_mapping, f, indent=2)
print('Saved: pair3_mitre_mapping.json')

## 15. Summary & Limitations

### What We Did
- Clustered **11,743 anomaly rows** using K-Means, GMM, and DBSCAN
- Selected optimal k via elbow/silhouette/BIC
- Evaluated with silhouette, Davies-Bouldin, Calinski-Harabasz
- Tested stability via Adjusted Rand Index across 10 seeds
- Profiled clusters and hypothesized MITRE ATT&CK mappings

### Key Limitations
1. **No ground-truth attack labels** — MITRE mapping is purely hypothetical
2. **Kaggle features ≠ HoneyBadger features** — TCP flags, byte counts, durations not captured by honeypot
3. **Clustering assumes attack types form distinct groups** — real attacks may be mixed/overlapping
4. **SSH subset is tiny (58 rows)** — cannot reliably profile SSH-specific attacks

### HoneyBadger Relevance
This exercise validates the **approach** of grouping attacks by behavioral patterns. For production use on HoneyBadger events:
- Need session-level aggregations (count, srv_rate, failed_logins) from `source_ip` + time windows
- MITRE tags from `backend/mitre.py` are rule-based, not learned
- A deployable model would need labeled honeypot data or transfer learning from this dataset